# Context & State

In a real app, there are two types of information involved:

- Information your Python server knows before the chat starts (e.g., user_id, account_balance, auth_token). The user didn't type this in; your backend database already has it.

- Information created during the chat conversation (e.g., the message history, or the status of a money transfer the bot is performing).

This difference is the entire reason Context and State exist.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Context

Context is basically the static backend data that the agent can only READ and cannot write/update.

For e.g: In a real world, suppose we're making a Banking AI Support Chatbot

When user says something like _"How much is my account balance?"_

Our agent gets to see the user's `account_number`, `account_balance`, `db_connection`, `user_id`, `language`, etc. as context.

This is context. And is given to the agent by our python backend database server on runtime.

We don't pass the context directly into the agent's prompt as it can hit token limits and the LLM might expose sensitive data. 

The way the agent can get access to context is through Tools, by a ToolRuntime object so that it only fetches what's needed. Not EVERYTHING.

In [25]:
class Database:
    account_balance : float = 25000

In [26]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class BankUserContext:
    db_connection : Optional[Database] = None
    user_id : str = "123"
    account_number : str = "SHAM06"

In [ ]:
from langchain.agents import create_agent

system_prompt = """
ANSWER VERY BRIEFLY. NO LENGTHY EXPLANATIONS.
Only to the point response.
Help users with their Banking operations.
"""

agent = create_agent(
    model="gpt-5-nano",
    context_schema=BankUserContext,
    system_prompt=system_prompt
)

In [15]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my account balance?")]},
    context=BankUserContext()
)

In [ ]:
from pprint import pprint

pprint(response) # it doesn't tell us about the account balance even tho we passed the context arguments
# this is because passing the context arguments doesn't automatically let the agent know the context
# we need to make tools to get access to particular information from the context

{'messages': [HumanMessage(content='What is my account balance?', additional_kwargs={}, response_metadata={}, id='64b2d444-1f0d-429a-9a93-aa995a47aa22'),
              AIMessage(content='I don’t have access to your accounts, so I can’t see your balance directly. If you tell me which bank or app you use, I can give you exact steps. In the meantime, here are quick ways to check your balance:\n\n- Mobile banking app:\n  - Open the app, sign in, go to Accounts or Balances, and select the account.\n- Online banking website:\n  - Log in, navigate to Accounts or Balances, and view your balance.\n- ATM:\n  - Insert your card, enter your PIN, choose Balance or View Accounts.\n- Phone banking:\n  - Use the bank’s self-service number (IVR or app) and follow prompts to check balance.\n- In person:\n  - Visit a branch and ask for your current balance.\n\nIf you tell me your bank/app, I’ll tailor the steps precisely. Also, I can help set up balance alerts or explain any recent transactions if you ha

## Accessing Context

In [27]:
# to get access to context we make tools with ToolRuntime objects

from langchain.tools import tool, ToolRuntime

@tool
def get_account_number(runtime: ToolRuntime) -> str:
    """Get the account number of the user"""
    return runtime.context.account_number

@tool
def get_user_id(runtime: ToolRuntime) -> str:
    """Get the user id of the user"""
    return runtime.context.user_id

@tool
def get_account_balance(runtime: ToolRuntime) -> int:
    """Get the account balance of the user"""
    return runtime.context.db_connection.account_balance

In [ ]:
system_prompt = """
ANSWER VERY BRIEFLY. NO LENGTHY EXPLANATIONS.
Only to the point response.
Help users with their Banking operations.
"""

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_account_number, get_user_id, get_account_balance],
    context_schema=BankUserContext,
    system_prompt=system_prompt
)

In [ ]:
# when I passed the BankUserContext() directly in the context= argument, and tried to see account balance,
# it gave me an error that there's no attribute 'account_balance'2.1_mcp.ipynb
# so to fix that, we initiliased the the Database class and BankUserContext class and passed the db object inside it

db = Database()

user_context = BankUserContext(db_connection=db)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my account balance?")]},
    context=user_context
)

pprint(response)

c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BankUserContext(db_connec...account_number='SHAM06'), input_type=BankUserContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BankUserContext(db_connec...account_number='SHAM06'), input_type=BankUserContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my account balance?', additional_kwargs={}, response_metadata={}, id='71e99038-154b-49d3-991b-803ae9c386ac'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 212, 'prompt_tokens': 191, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9qrJ0bqgZMSYHTvXXgVI7ROaCjFE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd6e6-6173-7910-9663-9d7e1b7abade-0', tool_calls=[{'name': 'get_account_balance', 'args': {}, 'id': 'call_o2jS7zq1idRRV6V4K1H50w3P', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 191, 'output_to

In [31]:
pprint(response["messages"][-1].content)

'Your account balance is 25,000.'


In [32]:
# we can pass the context values like this as well

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my account number?")]},
    context=BankUserContext(account_number="SHAM2005")
)

pprint(response)

c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BankUserContext(db_connec...count_number='SHAM2005'), input_type=BankUserContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BankUserContext(db_connec...count_number='SHAM2005'), input_type=BankUserContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my account number?', additional_kwargs={}, response_metadata={}, id='42bc08b2-8806-401f-8539-9f980f95ab0b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 148, 'prompt_tokens': 191, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9qwFwwjuvLxViw73kHYekYK98AGL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd6eb-091b-7cb2-9dfd-3ee6cbac760c-0', tool_calls=[{'name': 'get_account_number', 'args': {}, 'id': 'call_BoEBNOjGlDajvswg2HbGmeVS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 191, 'output_toke